# 30 Diffbot descarga textos

Al ejecutar la primera celda, se pide la autorización para acceder a Gdrive. A veces falla en el primer intento, volver a intentarlo.

In [31]:
# IMPORTS
import os
import re
import io
import time
import requests
from datetime import datetime
from dotenv import load_dotenv

import gspread
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload

import pandas as pd
import json
from pathlib import Path

# url_sin_detenidos = 'https://docs.google.com/spreadsheets/d/1xKAdnEC8HP4b5MJ-HrOGKqa7cXE-ue336RsWkIAhnYg/edit?gid=762130750#gid=762130750'

# EDG's copy for testing
url_sin_detenidos = 'https://docs.google.com/spreadsheets/d/1ZNzwwJXkg1nX2UmkXnuwocMBovUuRTc3OT6FxNx0usg/edit?usp=sharing'
url_folder = "https://drive.google.com/drive/folders/1i5Jl-Oho9k6PJPk4apsaUTwZbTCh8-oV?usp=sharing"

try:
    from google.colab import auth, userdata
    import google.auth

    # Running in Colab
    auth.authenticate_user()
    creds, _ = google.auth.default()
    client = gspread.authorize(creds)

    api_key = userdata.get('DIFFBOT_API_KEY')

except ImportError:
    from google.oauth2.service_account import Credentials

    # Running locally or outside Colab
    SCOPES = [
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive",
        'https://www.googleapis.com/auth/drive.file'
    ]

    creds = Credentials.from_service_account_file(
        "../secrets/credentials.json", scopes=SCOPES
    )
    client = gspread.authorize(creds)

    load_dotenv()  # Automatically finds .env file
    api_key = os.getenv('DIFFBOT_API_KEY')


spreadsheet = client.open_by_url(url_sin_detenidos)
drive_service = build('drive', 'v3', credentials=creds)

# Extract the folder ID from url_folder (the segment right after "folders/")
folder_id_match = re.search(r"/folders/([^/?]+)", url_folder)
folder_id = folder_id_match.group(1) if folder_id_match else None


### Dataframe con artículos (sin el texto) registrados por el RSS

In [26]:
# ws = spreadsheet.get_worksheet(0)
ws = spreadsheet.worksheet('INBOX')

# 1. Get raw cell values as a 2D list
raw_values = ws.get_all_values()

# 2. Define your own custom header names
headers = ['Fecha_deteccion', 'Medio', 'Titulo', 'Link', 'Keyword_detectada',
        'Fuente', 'Revisado', 'Validado', 'Observaciones', 'Estado_IA',
        'Palabras_detectadas', 'Puntaje', 'Puntaje_solo_titulo', 'otro_2']

# 3. Map values to records (skipping row 0 if row 0 has the old headers)
all_records = gspread.utils.to_records(headers, raw_values[1:])

# Example output
# for row in all_records[:2]:  # Print first 2 rows
#     print(row)

# {'Fecha_deteccion': 'desde aca ejecuté el nuevo código', 'Medio': '', 'Titulo': '', 'Link': '', 'Keyword_detectada': '', 'Fuente': '', 'Revisado': 'FALSE', 'Validado': '', 'Observaciones': '', 'Estado_IA': '', 'Palabras_detectadas': '', 'Puntaje': '', 'Puntaje_solo_titulo': '', 'otro_2': ''}
# {'Fecha_deteccion': '4/8/2026', 'Medio': 'Google News', 'Titulo': 'Facundo Moyano fue demorado en Belgrano: pelea de pareja y estupefacientes - andigital.com.ar', 'Link': 'https://news.google.com/rss/articles/CBMisAFBVV95cUxOMW5QRFdDeFRrZkVKbGFqTHljUnVMQkVSV1pRazQ4eXo3ajAwR1kzMDRWN2JaY01HOEdpeEp0TmFvQS04Z2VEMDdXQ3AyT29lemwzRHhTUURndUhrRDAtWU0tT3R1N1pmejd6c2hfQ0I5UnV2emlQdmthczZUX0FIbkRpY2NEY0Z0c0k4ZkJUNFhWMWZxdHF3ZTNndVR2ZDRkdU5MSW1PYmItU1kwd0lwUw?oc=5', 'Keyword_detectada': 'demorado', 'Fuente': 'https://news.google.com/rss/search?q=demorado%20CABA&hl=es-419&gl=AR&ceid=AR:es-419', 'Revisado': 'FALSE', 'Validado': 'PENDIENTE', 'Observaciones': '', 'Estado_IA': 'PROCESADO', 'Palabras_detectadas': 'POLICIA: policia, efectivo, uniformado, comisaria, policial, GNA, DIR | VICTIMA: demorado, victima | VIOLENCIA_POLICIAL: operativo | VIOLENCIA_GENERAL: incidente, golpe, violencia | CABA: ciudad de buenos aires, Belgrano, hospital pirovano', 'Puntaje': '12', 'Puntaje_solo_titulo': '', 'otro_2': ''}

# Convert records directly to DataFrame
df_sd = pd.DataFrame(all_records)

# Remove invalid rows: no Title
col = df_sd.columns[2]

# Filter out empty/whitespace strings and NaNs
df_sd = df_sd[df_sd[col].astype(str).str.strip().ne("") & df_sd[col].notna()]

# Reset index, keep the old index for reference, matching the row number in the gdrive sheet.
df_sd.reset_index(names="gdrive_index", inplace=True)

# Remove column "Medio", it's always equal to "Google News"
df_sd.drop(columns=["Medio"], inplace=True)

# Remove column "Fuente", it shows the search source URL, not the url for the article
df_sd.drop(columns=["Fuente"], inplace=True)

if "archivo" not in df_sd.columns:
    df_sd["archivo"] = pd.NA

# Últimos artículos
df_sd.iloc[-5:]

,gdrive_index,Fecha_deteccion,Titulo,Link,Keyword_detectada,Revisado,Validado,Observaciones,Estado_IA,Palabras_detectadas,Puntaje,Puntaje_solo_titulo,otro_2,archivo
402,405,10/9/2026,Video: un policía de la Ciudad mató a tiros a ...,https://news.google.com/rss/articles/CBMi8AFBV...,POLICIA + VIOLENCIA GENERAL,FALSE,PENDIENTE,,,POLICIA: policia | VIOLENCIA_GENERAL: mato,,,,<NA>
403,406,10/9/2026,Berisso: importante movilización contra la rep...,https://news.google.com/rss/articles/CBMiyAFBV...,POLICIA + POSIBLE VICTIMA + VIOLENCIA POLICIAL,FALSE,PENDIENTE,,,"POLICIA: policia, policial | POSIBLE_VICTIMA: ...",,,,<NA>
404,407,11/9/2026,El riesgo de los policías sin uniforme en las ...,https://news.google.com/rss/articles/CBMirwFBV...,POLICIA + POSIBLE VICTIMA,FALSE,PENDIENTE,,,POLICIA: policia | POSIBLE_VICTIMA: protesta,,,,<NA>
405,408,11/9/2026,Incidentes frente al Congreso: 26 detenidos y ...,https://news.google.com/rss/articles/CBMitwFBV...,POLICIA + VICTIMA + VIOLENCIA GENERAL + CABA,FALSE,PENDIENTE,,,POLICIA: policia | VICTIMA: detenido | VIOLENC...,,,,<NA>
406,409,12/9/2026,Citan a indagatoria al policía de la Ciudad qu...,https://news.google.com/rss/articles/CBMizwFBV...,POLICIA + VIOLENCIA GENERAL + CABA,FALSE,PENDIENTE,,,"POLICIA: policia | VIOLENCIA_GENERAL: asesino,...",,,,<NA>


El RSS devuelve artículos de cualquier fecha, no las últimas noticias. Acceder a algún artículo para verificar que la fecha es de años anteriores.

In [7]:
df_sd.loc[df_sd["gdrive_index"] == 3, "archivo"] = 'aaaaa.txt'

In [10]:
df_sd.iloc[1:5]

,gdrive_index,Fecha_deteccion,Titulo,Link,Keyword_detectada,Revisado,Validado,Observaciones,Estado_IA,Palabras_detectadas,Puntaje,Puntaje_solo_titulo,otro_2,archivo
1,2,4/8/2026,"Quién es Facundo Moyano, el dirigente sindical...",https://news.google.com/rss/articles/CBMixwFBV...,demorado,FALSE,PENDIENTE,,PROCESADO,,,,saltea filas consecutivas y excede tiempo de e...,<NA>
2,3,4/8/2026,“Marcha de la gorra” | Miles de personas movil...,https://news.google.com/rss/articles/CBMirwFBV...,gatillo,FALSE,PENDIENTE,,PROCESADO,,,,,<NA>
3,4,4/8/2026,Represión en Puente Pueyrredón: la UTEP denunc...,https://news.google.com/rss/articles/CBMizgFBV...,represión,FALSE,PENDIENTE,,PROCESADO,"POLICIA: policia, efectivo, policial, DIR, fue...",13,,,<NA>
4,5,4/8/2026,"Corridas, represión y detenidos en el Obelisco...",https://news.google.com/rss/articles/CBMiXkFVX...,represión,FALSE,PENDIENTE,,PROCESADO,,,,,<NA>


In [ ]:
print(df_sd.iloc[402]['Link'])

In [16]:
_file_counter = 0

def generar_nombre_archivo():
    """
    Generates a filename like YYYYMMDD_NNNNNN.txt using today's date
    and an incrementing counter (per run).
    """
    global _file_counter
    _file_counter += 1
    fecha = datetime.now().strftime("%Y%m%d")
    return f"{fecha}_{_file_counter:05d}.txt"

In [17]:
generar_nombre_archivo()

'20260912_00001.txt'

In [34]:
#SAVE FILE

def guardar_texto_en_drive(texto, nombre_archivo=None, folder_id=folder_id):
    """
    Saves `texto` as a .txt file inside the Google Drive folder `folder_id`.
    Returns the created file's id and name.
    """
    if nombre_archivo is None:
        nombre_archivo = generar_nombre_archivo()

    file_metadata = {
        "name": nombre_archivo,
        "parents": [folder_id],
    }
    media = MediaIoBaseUpload(
        io.BytesIO((texto or "").encode("utf-8")),
        mimetype="text/plain",
        resumable=False,
    )
    created_file = drive_service.files().create(
        body=file_metadata, media_body=media, fields="id, name"
    ).execute()

    return created_file


In [35]:
texto = "Espero que ande."
guardar_texto_en_drive(texto)

HttpError: <HttpError 403 when requesting https://www.googleapis.com/upload/drive/v3/files?fields=id%2C+name&alt=json&uploadType=multipart returned "Service Accounts do not have storage quota. Leverage shared drives (https://developers.google.com/workspace/drive/api/guides/about-shareddrives), or use OAuth delegation (http://support.google.com/a/answer/7281227) instead.". Details: "[{'message': 'Service Accounts do not have storage quota. Leverage shared drives (https://developers.google.com/workspace/drive/api/guides/about-shareddrives), or use OAuth delegation (http://support.google.com/a/answer/7281227) instead.', 'domain': 'usageLimits', 'reason': 'storageQuotaExceeded'}]">

## Diffbot

In [ ]:
# base_dir = Path.cwd()
# csv_path = base_dir / ".." / "data" / "Monitoreo noticias with articles.csv"
# output_csv_path = csv_path

# df = pd.read_csv(csv_path)
# print(f"Number of rows in DataFrame: {len(df)}")

# Rows to process from the DataFrame
# If not set, it will fetch all rows that do not have a diffbot response already.
n_i = 396
n_f = 399

if 
print(f"Processing rows {n_i} to {n_f} from DataFrame...")
print("Waiting 1 minute between requests to avoid hitting the rate limit...")

if "diffbot_response" not in df_sd.columns:
    df_sd["diffbot_response"] = pd.NA

url = f"https://api.diffbot.com/v3/article?token={api_key}"
headers = {}

subset = df_sd.iloc[n_i:n_f]


# Filter only rows that do not have a diffbot response already.
def needs_diffbot_response(value):
    if pd.isna(value):
        return True
    if not isinstance(value, str):
        return True
    text = value.strip()
    if not text:
        return True
    try:
        parsed = json.loads(text)
    except (ValueError, TypeError):
        return True
    return not (isinstance(parsed, dict) and "request" in parsed)

missing_response = subset["diffbot_response"].apply(needs_diffbot_response)

for index, row in subset[missing_response].iterrows():
# for index, row in subset.iterrows():
    print(f"Processing row {index} (ID = {row.get('ID')})...", end="")
    link = row.get("Link")
    if pd.isna(link) or not str(link).strip():
        print(f" skipping, missing link")
        continue

    try:
        params = {
            "url": link
        }
        response = requests.request("GET", url, params=params, headers=headers)
        print(f" status={response.status_code}")
        response_json = response.json()
        df_sd.at[index, "diffbot_response"] = json.dumps(response_json)
        # df_sd.to_csv(output_csv_path, index=False)
    except requests.RequestException as exc:
        print(f" request failed: {exc}")

    time.sleep(65)  # Sleep for 65 seconds to avoid hitting the rate limit


# print(f"Saved dataframe to {output_csv_path}")


In [ ]:
df_sd.iloc[395:400]['diffbot_response']

In [ ]:
obj = json.loads((df_sd.iloc[398]['diffbot_response']) or '{}')
json_obj = (obj.get('objects') or [{}])[0]
# print(json_obj.get('resolvedPageUrl'))
print(json_obj.get('text'))

# # Save to txt
# with open("output_399.txt", "w", encoding="utf-8") as f:
#     f.write(json_obj.get('text') or '')

In [ ]:
print(json_obj.get('resolvedPageUrl'))
